# SQLite-Backed AI Support Assistant with Tool Calling

## 1. Imports & Core Dependencies

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

## 2. Environment Setup & API Client Initialization

In [ ]:
load_dotenv(override=True)

openrouter_api_key= os.getenv("OPENROUTER_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")
ollama_api_key = "ollama"


openrouter_url = "https://openrouter.ai/api/v1"
gemini_url= "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url= "http://localhost:11434/v1"

gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key=ollama_api_key, base_url=ollama_url)


gemini_model = "gemini-3.6-flash"
gpt_model = "openai/gpt-4o-mini"
openrouter_model="mistralai/mistral-7b-instruct"
ollama_model = "llama3.2"

## 3. SQLite Database Setup & Schema Creation

In [ ]:
import sqlite3

In [ ]:
DB = "store.db"
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS categories (category_key TEXT PRIMARY KEY, url TEXT, description TEXT)')

## 4. System Prompt & Support Bot Persona


In [ ]:
SYSTEM_MESSAGE = """
You are a helpful support bot for Palm Homoeo Store, a homoeopathic medicines store.
When a user describes a symptom, use the `get_category_link` tool to fetch the product link.
Always share the link clearly and mention shop contacts:
- Phone: +92 423 002 0058
- WhatsApp: +92 343 44208020
"""

## 5. Data Seeding — Populate SQLite Categories


In [ ]:
categories_data = [
        ("hair", "https://palmhomoeostore.com/product-category/symptoms/hair-tonics-symptoms/", "Hair tonics, hair fall, dandruff"),
        ("bone", "https://palmhomoeostore.com/product-category/symptoms/bone-strength/", "Bone strength, joint pain"),
        ("fever", "https://palmhomoeostore.com/product-category/symptoms/fever-symptoms/", "Fever and high temperature"),
        ("acidity", "https://palmhomoeostore.com/product-category/symptoms/homeopathic-medicines-for-acidity/", "Acidity, stomach gas, indigestion"),
        ("acne", "https://palmhomoeostore.com/product-category/symptoms/acne-pimples/", "Acne and pimples"),
        ("sciatica", "https://palmhomoeostore.com/product-category/symptoms/sciatica-symptoms/", "Sciatica nerve pain"),
        ("vertigo", "https://palmhomoeostore.com/product-category/symptoms/vertigo-symptoms/", "Vertigo and dizziness"),
        ("nausea", "https://palmhomoeostore.com/product-category/symptoms/nausea-and-vomiting/", "Nausea and vomiting"),
        ("asthma", "https://palmhomoeostore.com/product-category/symptoms/asthma/", "Asthma and breathing issues"),
        ("cough", "https://palmhomoeostore.com/product-category/symptoms/cough-symptoms/", "Cough and throat problems"),
        ("nutrition", "https://palmhomoeostore.com/product-category/symptoms/whole-familys-nutrition/", "Whole family nutrition"),
        ("personal_care", "https://palmhomoeostore.com/product-category/personal-care/", "Personal care products"),
        ("male", "https://palmhomoeostore.com/?s=male&post_type=product", "Male health products"),
        ("female", "https://palmhomoeostore.com/?s=female&post_type=product", "Female health products"),
        ("children", "https://palmhomoeostore.com/?s=children&post_type=product", "Children health products")
    ]

for key, url, desc in categories_data:
        cursor.execute('INSERT OR REPLACE INTO categories VALUES (?, ?, ?)', (key, url, desc))
conn.commit()

## 6. Backend Tool Function — SQLite Category Lookup


In [ ]:
def get_category_link(category_key):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT url, description FROM categories WHERE category_key = ?', (category_key.lower(),))
        row = cursor.fetchone()
        if row:
            return f"URL: {row[0]} | Details: {row[1]}"
        return "No specific category link found. Direct them to https://palmhomoeostore.com/"


## 7. Tool Schema Definition (Function Calling Specification)


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_category_link",
            "description": "Get the website URL for a specific product category or symptom.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category_key": {
                        "type": "string",
                        "description": "The symptom key: 'hair', 'bone', 'fever', 'acidity', 'acne', 'sciatica', 'vertigo', 'nausea', 'asthma', 'cough', 'nutrition', 'personal_care', 'male', 'female', 'children'",
                    }
                },
                "required": ["category_key"],
                "additionalProperties": False
            }
        }
    }
]

## 8. Tool Execution Handler


In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_category_link":
            args = json.loads(tool_call.function.arguments)
            result = get_category_link(args.get('category_key'))
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })
    return responses

## 9. Main Chat Loop — Multi-Turn Tool Calling Engine


In [ ]:
def chat(message, history):
    formatted_history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_MESSAGE}] + formatted_history + [{"role": "user", "content": message}]
    
    response = openrouter.chat.completions.create(model=openrouter_model, messages=messages, tools=tools)
    
    while response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        tool_responses = handle_tool_calls(assistant_msg)
        messages.append(assistant_msg)
        messages.extend(tool_responses)
        response = openrouter.chat.completions.create(model=openrouter_model, messages=messages, 
        tools=tools)

    return response.choices[0].message.content

## 10. User Interface — Gradio UI Launch


In [ ]:
gr.ChatInterface(fn=chat, title="Palm Homoeo Store Assistant").launch()